✅ Step 1: Install Required Libraries

In [ ]:
pip install transformers sentence-transformers faiss-cpu accelerate gradio

✅ Step 2: Create a Small Knowledge Base
Create a folder called knowledge_base/ and add some .txt files there.

Example:

knowledge_base/company_policy.txt

knowledge_base/faq.txt

knowledge_base/about_us.txt

Each file can just have some random paragraphs of text.

✅ Step 3: Split Documents into Chunks

We split long documents into smaller pieces (300–500 words).

In [1]:
import os

def load_documents(directory):
    documents = []
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            with open(os.path.join(directory, filename), "r", encoding="utf-8") as file:
                text = file.read()
                documents.append(text)
    return documents

docs = load_documents("knowledge_base/")

✅ Step 4: Embed the Documents (Vectorization)

Now, convert those chunks into embeddings:

In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model

embeddings = embedder.encode(docs, convert_to_tensor=True)

✅ Step 5: Store Embeddings in FAISS

FAISS lets you quickly search similar vectors.

In [4]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity

index.add(embeddings.cpu().numpy())  # Add embeddings

✅ Step 6: Search for Relevant Chunks (Retriever)

In [5]:
def retrieve_relevant_docs(query, k=3):
    query_embedding = embedder.encode([query], convert_to_tensor=True).cpu().numpy()
    D, I = index.search(query_embedding, k)  # D = distances, I = indices
    results = [docs[i] for i in I[0]]
    return results

✅ Step 7: Generate Answer Using LLM

Use a Hugging Face text model TinyLlama

In [6]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto",
    trust_remote_code=True  # needed for some models
)

Some parameters are on the meta device because they were offloaded to the cpu and disk.
Device set to use cpu


In [7]:
def generate_answer(query):
    context = retrieve_relevant_docs(query)
    top_context = context[0]

    prompt = f"""You are a helpful assistant.
Use ONLY the provided context to answer briefly and accurately. 
Do not invent anything beyond the given context.
Answer in 1-2 short sentences maximum.

Context:
{top_context}

User's Question: {query}
"""

    response = generator(prompt, max_new_tokens=50)[0]['generated_text']
    return response


✅ Step 8: Build a Chat UI (Gradio)

In [8]:
import gradio as gr

def chat(query):
    answer = generate_answer(query)
    return answer

gr.Interface(fn=chat, inputs="text", outputs="text", title="🧠 My RAG Chatbot").launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
